In [1]:
import modal

app = modal.App("gemma4-chartqa-test")

image = (
    modal.Image.debian_slim(python_version="3.12")
    .pip_install(
        "torch",
        "torchvision",
        "transformers>=5.10.1",
        "peft",
        "accelerate",
        "pillow",
        "hf_transfer",
        "datasets",
    )
    .env({"HF_HUB_ENABLE_HF_TRANSFER": "1"})
)

BASE_MODEL = "google/gemma-4-31B-it"
ADAPTER = "Arnav-Gr0ver/Gemma4-31B-IT-ChartQA-Adapted"

hf_secret = modal.Secret.from_name("huggingface-secret")

In [9]:
@app.function(image=image, gpu="A100-80GB", timeout=87600, secrets=[hf_secret])
def eval_chartqa(n_samples: int = 500):
    import torch
    import time
    import json
    from transformers import AutoModelForImageTextToText, AutoProcessor
    from peft import PeftModel
    from datasets import load_dataset

    processor = AutoProcessor.from_pretrained(BASE_MODEL)
    base_model = AutoModelForImageTextToText.from_pretrained(
        BASE_MODEL, torch_dtype=torch.bfloat16, device_map="cuda",
    )
    model = PeftModel.from_pretrained(base_model, ADAPTER)
    model.eval()

    print("Active adapters:", model.active_adapters())

    ds = load_dataset("HuggingFaceM4/ChartQA", split="test")
    if n_samples:
        ds = ds.select(range(min(n_samples, len(ds))))

    results = []
    start_time = time.time()

    for i, ex in enumerate(ds):
        ex_start = time.time()

        img = ex["image"].convert("RGB")
        question = ex["query"]
        gold = ex["label"][0] if isinstance(ex["label"], list) else ex["label"]

        messages = [{"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": question},
        ]}]

        inputs = processor.apply_chat_template(
            messages, add_generation_prompt=True, enable_thinking=True,
            tokenize=True, return_dict=True, return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False)

        full_text = processor.decode(
            output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False,
        )

        reasoning, final = "", full_text
        if "<|channel>thought" in full_text and "<channel|>" in full_text:
            after_open = full_text.split("<|channel>thought", 1)[1]
            reasoning, rest = after_open.split("<channel|>", 1)
            reasoning = reasoning.lstrip("\n")
            final = rest

        for tok in ["<|turn>model", "<turn|>", "<|channel>thought", "<channel|>"]:
            final = final.replace(tok, "")
        final = final.strip()

        ex_time = time.time() - ex_start

        result = {
            "question": question,
            "gold": gold,
            "prediction": final,
            "reasoning": reasoning,
            "raw_output": full_text,
            "time_sec": ex_time,
        }
        results.append(result)

        print(f"[{i+1}/{len(ds)}] time={ex_time:.1f}s gold={gold!r} pred={final!r}")

    total_time = time.time() - start_time
    print(f"\nTotal: {total_time:.1f}s for {len(ds)} examples, avg {total_time/len(ds):.1f}s/example")

    return {"n": len(ds), "total_time": total_time, "results": results}

In [16]:
import json

with modal.enable_output():
    with app.run():
        eval_result = eval_chartqa.remote(n_samples=500)

print(f"Ran {eval_result['n']} examples in {eval_result['total_time']:.1f}s")

with open("chartqa_results_from_adapted.json", "w") as f:
    json.dump(eval_result, f, indent=2)